Cell 1: Environment & Imports

In [5]:
import numpy as np
import pandas as pd
import yfinance as yf
import cvxpy as cp
from hmmlearn import hmm
import matplotlib.pyplot as plt
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

ModuleNotFoundError: No module named 'pandas'

Cell 2: Data Acquisition

In [ ]:
# Pulling investable assets: Equity (Nifty 50), Gold, and Bonds
investable_tickers = ["^NSEI", "GOLDBEES.NS", "LICNETFGSC.NS"] 
price_data = yf.download(investable_tickers, start="2015-01-01", end="2024-01-01", progress=False)['Close']
price_data = price_data.ffill().dropna()

# Pulling VIX for feature engineering
vix_data = yf.download("^INDIAVIX", start="2015-01-01", end="2024-01-01", progress=False)['Close']

Cell 3: Feature Engineering (Safe Z-Scoring)

In [ ]:
# Calculating daily log returns
multi_asset_returns = np.log(price_data).diff().dropna()
nifty_ret = multi_asset_returns["^NSEI"]

# Creating expanding window features to prevent lookahead bias
vol_21d = nifty_ret.rolling(21).std() * np.sqrt(252)

feature_data = pd.DataFrame(index=multi_asset_returns.index)
feature_data['vol_zscore'] = (vol_21d - vol_21d.expanding(min_periods=63).mean()) / vol_21d.expanding(min_periods=63).std()
feature_data['vix_zscore'] = (vix_data - vix_data.expanding(min_periods=63).mean()) / vix_data.expanding(min_periods=63).std()
feature_data = feature_data.dropna()

# Align indices
common_dates = multi_asset_returns.index.intersection(feature_data.index)
multi_asset_returns = multi_asset_returns.loc[common_dates]
feature_data = feature_data.loc[common_dates]

Cell 4: The Convex Optimizer

In [ ]:
# 1. The Bulletproof Optimizer
def optimize_portfolio(returns_matrix, regime_label):
    n_assets = returns_matrix.shape[1]
    mu = returns_matrix.mean().values
    
    # Add epsilon to diagonal for strict positive definiteness
    Sigma = returns_matrix.cov().values + np.eye(n_assets) * 1e-8 
    
    w = cp.Variable(n_assets)
    constraints = [cp.sum(w) == 1, w >= 0.05, w <= 0.80]
    
    # Mathematically promise CVXPY the matrix is safe using psd_wrap
    risk = cp.quad_form(w, cp.psd_wrap(Sigma))
    
    if regime_label == 1:
        objective = cp.Minimize(risk)
    elif regime_label == 2:
        risk_aversion = 50.0 
        objective = cp.Maximize(mu @ w - risk_aversion * risk)
    else:
        risk_aversion = 150.0
        objective = cp.Maximize(mu @ w - risk_aversion * risk)

    prob = cp.Problem(objective, constraints)
    
    try:
        prob.solve(solver=cp.SCS)
        if prob.status in ["optimal", "optimal_inaccurate"] and w.value is not None:
            clean_weights = np.clip(w.value, 0.05, 0.80)
            return clean_weights / clean_weights.sum()
        else:
            return np.ones(n_assets) / n_assets
    except:
        return np.ones(n_assets) / n_assets

Cell 5: The Walk-Forward Engine

In [ ]:
# 2. Reset the weights list to clear old Jupyter memory!
target_weights = []
window_size = 252 * 2  

# 3. Walk-Forward Engine
for i in tqdm(range(window_size, len(feature_data)), desc="Walk-Forward Engine"):
    
    train_features = feature_data.iloc[i - window_size : i]
    train_returns = multi_asset_returns.iloc[i - window_size : i]
    
    if i % 5 == 0:
        model = hmm.GaussianHMM(n_components=3, covariance_type="full", n_iter=50, random_state=42)
        model.fit(train_features.values)
        
        state_variances = np.array([np.diag(model.covars_[s])[0] for s in range(3)])
        sorted_states = np.argsort(state_variances)
        
        bull_state = sorted_states[0]    
        bear_state = sorted_states[1]    
        crisis_state = sorted_states[2]  
    
    current_day_features = feature_data.iloc[i:i+1].values
    raw_regime = model.predict(current_day_features)[0]
    
    if raw_regime == crisis_state:
        current_regime = 1
    elif raw_regime == bull_state:
        current_regime = 2
    else:
        current_regime = 0
        
    optimal_weights = optimize_portfolio(train_returns, current_regime)
    target_weights.append(optimal_weights)

Cell 6: Backtesting & Benchmarking

In [ ]:
# 4. Backtest Math
test_returns = multi_asset_returns.iloc[window_size:window_size+len(target_weights)]
weights_df = pd.DataFrame(target_weights, index=test_returns.index, columns=test_returns.columns)

implemented_weights = weights_df.shift(1).dropna()
actual_simple_returns = np.exp(test_returns.loc[implemented_weights.index]) - 1

gross_portfolio_returns = (implemented_weights * actual_simple_returns).sum(axis=1)
turnover = implemented_weights.diff().abs().sum(axis=1)
transaction_cost_bps = 0.0010 
costs = turnover * transaction_cost_bps

net_portfolio_returns = gross_portfolio_returns - costs
equity_curve = (1 + net_portfolio_returns).cumprod()

# 5. Metrics & Benchmark Calculation
def calculate_metrics(eq_curve, daily_ret):
    total_ret = eq_curve.iloc[-1] - 1
    ann_ret = (1 + total_ret) ** (252 / len(eq_curve)) - 1
    ann_vol = daily_ret.std() * np.sqrt(252)
    sharpe = ann_ret / ann_vol
    
    rolling_max = eq_curve.cummax()
    drawdown = (eq_curve - rolling_max) / rolling_max
    max_dd = drawdown.min()
    calmar = ann_ret / abs(max_dd)
    
    return pd.Series({
        "Total Return": f"{total_ret * 100:.2f}%",
        "Ann. Return": f"{ann_ret * 100:.2f}%",
        "Ann. Volatility": f"{ann_vol * 100:.2f}%",
        "Sharpe Ratio": round(sharpe, 2),
        "Max Drawdown": f"{max_dd * 100:.2f}%",
        "Calmar Ratio": round(calmar, 2)
    })

# Dynamically map the benchmark weights to whatever order yfinance alphabetized your columns
bm_weights_dict = {"^NSEI": 0.60, "LICNETFGSC.NS": 0.40, "GOLDBEES.NS": 0.0}
bm_weights_array = np.array([bm_weights_dict.get(col, 0) for col in test_returns.columns])

benchmark_simple_returns = (actual_simple_returns * bm_weights_array).sum(axis=1)
benchmark_equity = (1 + benchmark_simple_returns).cumprod()

print("\n--- Dynamic Regime-Shift Strategy ---")
print(calculate_metrics(equity_curve, net_portfolio_returns))

print("\n--- Static 60/40 Benchmark ---")
print(calculate_metrics(benchmark_equity, benchmark_simple_returns))